# Stage 4 Lab — ANSWER KEY (facilitator only)
Each bug: the fix, the one-line change, and the sentence to say out loud.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline

## BUG 1 fix — use `@` not `*`
`*` is element-wise; `@` is matrix multiplication. A² in a polynomial means A@A.

In [ ]:
A = np.array([[2,1],[0,3]]); I = np.eye(2, dtype=int)
pA = A @ A + 2*A + I          # FIX: @ instead of *
print(pA)                      # top-right now 7
# Say: "Star multiplies entry-by-entry. At-sign is real matrix multiply.
#       A squared in a polynomial always means A @ A."

## BUG 2 fix — turn the intercept back on
The line has a constant (+1). Without an intercept the model is forced through
the origin and can't fit it.

In [ ]:
X=np.array([[1.],[2.],[3.],[4.]]); y=np.array([3.,5.,7.,9.])
model=LinearRegression(fit_intercept=True).fit(X,y)   # FIX
print(round(model.predict([[1.0]])[0],3))              # 3.0
# Say: "fit_intercept=False forces the line through (0,0). Our data needs a
#       +1 offset, so the model must be allowed to learn a constant."


## BUG 3 fix — handle_unknown='ignore'
Unseen categories should encode as all-zeros, not crash.

In [ ]:
train=pd.DataFrame({"city":["Fergana","Tashkent","Namangan"]})
new=pd.DataFrame({"city":["Andijan"]})
enc=OneHotEncoder(handle_unknown="ignore").fit(train)   # FIX
print(enc.transform(new).toarray())                     # [[0. 0. 0.]]
# Say: "In production new categories always show up. 'ignore' encodes them as
#       all zeros instead of throwing. 'error' is what crashed us."


## BUG 4 fix — pipeline, so the scaler fits inside each fold
The leaky version scales all data before CV, leaking test stats. A pipeline
re-fits the scaler on only the training part of every fold.

In [ ]:
rng=np.random.RandomState(0)
X=rng.rand(200,5); y=(X[:,0]>0.5).astype(int)

pipe=make_pipeline(StandardScaler(), LogisticRegression())   # FIX
clean=cross_val_score(pipe, X, y, cv=5)
print("clean CV accuracy:", round(clean.mean(),3))
# Say: "Fitting the scaler on all data lets it peek at the test folds — that's
#       leakage. The pipeline guarantees the scaler only ever sees training
#       rows. Rule: split first, fit on train, transform both."
